# DWBI Week 2 — DW Lifecycle & Architecture
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

Case: **PakFreight** — three transport modes, three files, three totals, zero trust.
The week ends with one trusted table: `Shipments`, 34 rows, 578,650 kg.

## Task 3 — Data Detective

Workbook deliverable: `PakFreight_Activity_su92-bsdsm-f23-018.xlsx` — every yellow cell filled,
every check ✔ (verified with a formula engine, not by overtyping).

### 1. The problem log (12 entries, faults in all three files)

| # | File | Where | Problem | Report effect | Fix |
|---|---|---|---|---|---|
| 1 | Marine | `Ship_Date` | Text dates `dd/mm/yyyy` | US locale reads 03/02 as Mar 2; 13/02 errors | UK-locale date conversion in staging |
| 2 | Rail | `Weight (Tons)` | Weights in **tons**, not kg | Totals 1,000× too small — Finance's 283.70 t | `WeightKG = tons × 1000` |
| 3 | Rail | Rows 1–2 | Two title rows above the header | Auto headers grab the wrong row | Remove top 2, promote row 3 |
| 4 | Rail | `Consignment No` | Numeric IDs (45012) vs text IDs | Join/append type mismatch | Prefix `RL-`, store as text |
| 5 | Rail + Marine + Road | `Status` | Mixed case: Delivered/delivered/DELAYED | GROUP BY splits one status into three | TRIM + PROPER |
| 6 | Marine | `Status` | Leading/trailing spaces (`'DELIVERED '`) | Invisible mismatch, groups apart | TRIM |
| 7 | Road | `trip_id` | Lowercase IDs (`rd-501`) | Breaks the ID contract | UPPER → `RD-501` |
| 8 | Road | `rd-505` ×2 | Exact duplicate row | Double-counts 6,100 kg — Sales' 584.75 t | Dedupe, keep first |
| 9 | Road | Row 15 | Blank row | Null "shipment" in the model | Remove blank rows |
| 10 | Road | TOTAL row | Summary row, not a shipment | +83,000 kg — Operations' 661.65 t | Filter out; never load aggregates |
| 11 | All three | Column names | `Shipment_ID` / `Consignment No` / `trip_id` | Append by name creates ghost columns | Rename all to the contract |
| 12 | Road | `trip_date` | Text `'2026-02-02'` vs rail's real dates | Sorts as text; type clash | `DATEVALUE` in staging |

### 2. The Stovepipe Game — reproducing all four totals

In [ ]:
import pandas as pd
m = pd.read_csv("data/marine_shipments.csv")
r = pd.read_excel("data/rail_shipments.xlsx", header=2)
rd = pd.read_csv("data/road_shipments.csv", dtype=str)
rd["w"] = pd.to_numeric(rd["weight_kg"], errors="coerce")

marine = m["Weight_KG"].sum()                       # 206,500
rail_kg = (r["Weight (Tons)"] * 1000).sum()          # 295,250
rail_tons = r["Weight (Tons)"].sum()                 # 295.25  <- Finance's mistake
road_raw = rd["w"].sum()                             # 166,000 (incl. dup + TOTAL)

ops   = marine + rail_kg + (road_raw - 6100)         # removed dup, KEPT total
fin   = marine + rail_tons + (road_raw - 83000 - 6100)  # road cleaned, tons AS kg
sales = marine + rail_kg + (road_raw - 83000)        # removed TOTAL, KEPT dup
truth = marine + rail_kg + (road_raw - 83000 - 6100)

for name, v in [("Operations", ops), ("Finance", fin), ("Sales", sales), ("Staging (truth)", truth)]:
    print(f"{name:16s} {v:>10,.0f} kg = {v/1000:.2f} t")
print("\nWhy all three believed they were right: each summed their own file with their own")
print("private assumption, and with no shared staging there was no single rule to contradict them.")

![Four totals](charts/t3_stovepipes.png)

### 3. Staging in the workbook

Each staging sheet keeps RAW on the left (copied exactly, never edited) and the contract columns
on the right as **formulas** — e.g. Marine `ShipDate =DATE(RIGHT(B5,4),MID(B5,4,2),LEFT(B5,2))`
(the dd/mm/yyyy trap), Rail `ShipmentID ="RL-"&A5` and `WeightKG =E5*1000`, Road
`Keep? =IF(OR(A5="",A5="TOTAL",COUNTIF($A$5:A5,A5)>1),"N","Y")` with a named reason per dropped row.

### 4. Reconciliation — every check ✔

| Source | Raw | Removed | Loaded | Weight (kg) |
|---|---|---|---|---|
| Marine | 12 | 0 | 12 | 206,500 |
| Rail | 10 | 0 | 10 | 295,250 |
| Road | 15 | 3 | 12 | 76,900 |
| **Total** | **37** | **3** | **34** | **578,650** |

Status: Delivered 22 · In Transit 8 · Delayed 4. Date trap: MR-1003 = 03-Feb-2026 ✔, MR-1012 = 13-Feb-2026 ✔.

### 5. Lineage of 578,650 kg

![Lineage](charts/t3_lineage.png)

In words: `marine_shipments.csv` (12 rows, UK-locale dates, TRIM+PROPER status) → `marine_stg` (206,500 kg);
`rail_shipments.xlsx` (skip 2 title rows, tons × 1000, `RL-` prefix) → `rail_stg` (295,250 kg);
`road_shipments.csv` (UPPER IDs, drop duplicate `rd-505` / blank row / TOTAL row — each named) → `road_stg` (76,900 kg);
append → `Shipments` (34 × 8); measure `SUM(WeightKG)` → card **578,650 kg**.

### 6. Kimball bus matrix

![Kimball bus matrix](charts/t3_busmatrix.png)

| Process | Date | Customer | Route | Vehicle | Driver | Station/Port | Commodity |
|---|---|---|---|---|---|---|---|
| Shipments | X | X | X | X | | X | X |
| Billing & Invoicing | X | X | X | | | X | X |

Billing reuses the conformed **Date** and **Route** dimensions from Shipments — same keys, same meaning,
so cost and revenue join without reconciliation.

### The question to answer

One control so a TOTAL row can never reach a report: a **row-type guard in staging** — reject any row
whose ID is null, blank, or matches a non-shipment pattern (e.g. `TOTAL`), and add a reconciliation
check that fails the load if the staged row count ≠ raw count − named removals. The mistake then breaks
the pipeline loudly instead of inflating a report silently.